# DOWNLOADING DATA
For this project I only downloaded the flux, the flux error, and the bad pixel mask for inputs, and the effective temperature, log surface gravity, and metallicity for pontential outputs to classify the stars.

In [2]:
import pandas as pd
import numpy as np
import requests
import os
from astropy.io import fits
import io
import time

In [3]:
# Shuffling data for machine learning
df = pd.read_csv('cleaned_data.csv')
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_shuffled.to_csv('cleaned_data_shuffled.csv', index=False)

In [4]:
# CONSTANTS
base_url = "https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars"
csv_path = 'cleaned_data_shuffled.csv'
processed_data_path = 'processed_spectra.npy'
labels_path = 'processed_labels.npy'

df = pd.read_csv(csv_path)

# resume after pausing
if os.path.exists(processed_data_path):
    all_spectra = list(np.load(processed_data_path))
    start_idx = len(all_spectra)
    print(f"Resuming from star index {start_idx}...")
else:
    all_spectra = []
    start_idx = 0

# for faster downloading
session = requests.Session()

for idx, row in df.iloc[start_idx:].iterrows():
    start_time = time.time()
    url = f"{base_url}/{row['TELESCOPE']}/{row['FIELD']}/{row['FILE']}"
    
    try:
        r = session.get(url, timeout=10, stream=True) 
        
        if r.status_code == 200:
            with fits.open(io.BytesIO(r.content)) as file:
                flux = file[1].data[0]
                error = file[2].data[0]
                mask = file[3].data[0]
                
                good = (mask == 0) & (error > 0)
                med = np.nanmedian(flux[good])
                
                spec = np.stack([
                    np.where(good, flux/med, 1.0),
                    np.where(good, error/med, 10.0)
                ]).astype(np.float32)
                
                all_spectra.append(spec)
            
            elapsed = time.time() - start_time
            print(f"[{idx}] Done in {elapsed:.2f}s | {row['FILE']}")

        else:
            print(f"[{idx}] FAILED: Status {r.status_code} for {row['FILE']}")

        # Save every 100 stars
        if (idx + 1) % 100 == 0:
            print(f"--- Saving at {idx + 1} stars ---")
            np.save(processed_data_path, np.array(all_spectra))
            labels = df.iloc[0:len(all_spectra)][['RV_TEFF', 'RV_FEH', 'RV_LOGG']].values
            np.save(labels_path, labels.astype(np.float32))

    except Exception as ex:
        print(f"[{idx}] ERROR: {ex}")

# Final Save
print("Finalizing all files...")
final_data = np.array(all_spectra, dtype=np.float32)
np.save(processed_data_path, final_data)

final_labels = df.iloc[0:len(all_spectra)][['RV_TEFF', 'RV_FEH', 'RV_LOGG']].values
np.save(labels_path, final_labels.astype(np.float32))

print(f"Done! Final dataset shape: {final_data.shape}")

Resuming from star index 10188...
[10188] Done in 1.45s | asStar-dr17-2M05295780-6855190.fits
[10189] Done in 0.54s | apStar-dr17-2M01013463+1201299.fits
[10190] Done in 0.93s | apStar-dr17-2M04014146-0511255.fits
[10191] Done in 0.89s | apStar-dr17-2M06162436-0203261.fits
[10192] Done in 1.08s | apStar-dr17-2M23182422+1156508.fits
[10193] Done in 0.88s | apStar-dr17-2M05515977+2034023.fits
[10194] Done in 1.37s | apStar-dr17-2M17230885+4912009.fits
[10195] Done in 1.62s | asStar-dr17-2M12110216-5042072.fits
[10196] Done in 3.92s | apStar-dr17-2M19535064+1849075.fits
[10197] Done in 2.01s | apStar-dr17-2M01544541+3504476.fits
[10198] Done in 2.70s | apStar-dr17-2M07423695+1007286.fits
[10199] Done in 5.67s | asStar-dr17-2M14173282-4849243.fits
--- Saving at 10200 stars ---
[10200] Done in 3.59s | asStar-dr17-2M06400568-6838497.fits
[10201] Done in 3.58s | apStar-dr17-2M12133263+5142333.fits
[10202] Done in 3.85s | apStar-dr17-2M19281062+4045393.fits
[10203] Done in 2.81s | apStar-dr17-